# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [1]:
import pandas as pd
import numpy as np

In [2]:
x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

print("Comparing Ridge Regression Weights for different Alphas:\n" + "-"*60)

# Loop through the list of alphas
for alpha in alphas:
    # Calculate weights
    w = np.linalg.inv(x.T * x + alpha * I) * x.T * y
    w = np.asarray(w).ravel() # Convert matrix back to standard 1D array for easier printing
    
    # Compare the results
    print(f"Alpha: {alpha:>7} | Intercept (w0): {w[0]:>8.4f} | Slope (w1): {w[1]:>8.4f}")

Comparing Ridge Regression Weights for different Alphas:
------------------------------------------------------------
Alpha:   0.001 | Intercept (w0): -179.5263 | Slope (w1):   1.6102
Alpha:    0.01 | Intercept (w0): -167.8553 | Slope (w1):   1.5442
Alpha:     0.1 | Intercept (w0): -101.7240 | Slope (w1):   1.1698
Alpha:     1.0 | Intercept (w0): -20.5904 | Slope (w1):   0.7105
Alpha:    10.0 | Intercept (w0):  -2.2911 | Slope (w1):   0.6069
Alpha:   100.0 | Intercept (w0):  -0.2287 | Slope (w1):   0.5951


## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results. 

In [3]:
import numpy as np

def sgd(x, y, alpha, learning_rate=1e-5, epochs=10):
    # Initialize weights to zeros
    w = np.zeros((x.shape[1], 1))
    
    for _ in range(epochs):
        # Pick a random sample index for Stochastic Gradient Descent
        idx = np.random.randint(x.shape[0])
        xi = x[idx]
        yi = y[idx]
        
        # Calculate prediction error
        error = (xi * w)[0, 0] - yi[0]
        
        # Subgradient of the L1 penalty
        grad_l1 = np.zeros((x.shape[1], 1))
        grad_l1[1:] = alpha * np.sign(w[1:])
        
        # Update weights
        w = w - learning_rate * (xi.T * error + grad_l1)
        
    return w

In [4]:
import numpy as np
from sklearn.linear_model import Lasso

x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15,1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15,1)

x_mat = np.asmatrix(np.c_[np.ones((15,1)),x])

I = np.identity(2)
alpha = 0.1 

# custom SGD
w = sgd(x_mat, y, alpha) 
w = np.asarray(w).flatten()

# scikit-learn for comparison
lasso_sk = Lasso(alpha=alpha)
lasso_sk.fit(x, y)

print("--- Lasso Regression Results ---")
print(f"Manual SGD Intercept : {w[0]:>10.4f} | Slope: {w[1]:>8.4f}")
print(f"Scikit-Learn Intercept: {lasso_sk.intercept_[0]:>9.4f} | Slope: {lasso_sk.coef_[0]:>8.4f}")

--- Lasso Regression Results ---
Manual SGD Intercept :     0.0026 | Slope:   0.4514
Scikit-Learn Intercept: -180.8579 | Slope:   1.6178


As expected, the manual SGD implementation yields significantly different weights compared to the scikit-learn baseline. Because the assignment instructions specifically limited the SGD training to exactly 10 epochs on unscaled data, the algorithm only took a few small steps away from the initial zero weights and did not have enough iterations to converge. The scikit-learn model runs to full convergence using a highly optimized algorithm, which explains the large difference in the final intercept and slope.

## 3. Extend the Fisher's classifier

Please extend the targets of the ``iris_data`` variable and use it as the $y$.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris_data = load_iris()
iris_df = pd.DataFrame(iris_data.data,columns=iris_data.feature_names)
iris_df.head()

x = iris_df[['sepal length (cm)', 'sepal width (cm)']].values # change here
y = iris_data.target.reshape(-1, 1) # change here

dataset_size = np.size(x)

mean_x, mean_y = np.mean(x), np.mean(y)

SS_xy = np.sum(y * x) - dataset_size * mean_y * mean_x
SS_xx = np.sum(x * x) - dataset_size * mean_x * mean_x

a = SS_xy / SS_xx
b = mean_y - a * mean_x

y_pred = a * x + b

print(f"a: {a}")
print(f"b: {b}")
print(f"y_pred (first 5 rows):\n{y_pred[:5]}")

a: 0.07914567945747226
b: 0.6477753445210959
y_pred (first 5 rows):
[[1.05141831 0.92478522]
 [1.03558917 0.88521238]
 [1.01976004 0.90104152]
 [1.01184547 0.89312695]
 [1.04350374 0.93269979]]
